In [ ]:
import os
import copy
import time
import random
import pandas as pd
import numpy as np

import torch
from torch_geometric.data import HeteroData
from torch_frame.data import Dataset
from torch_frame import stype
numerical_type = stype.numerical
categorical_type = stype.categorical
embedding_type = stype.embedding
time_type = stype.timestamp
multi_categorical_type = stype.multicategorical

In [2]:
n_samples = 10

# Numerical column
num_1 = np.random.randint(0, 100, size=n_samples)
num_2 = np.random.randint(0, 3, size=n_samples)

# Categorical column
cat_1 = np.random.choice(['Type 1', 'Type 2', 'Type 3'], size=n_samples)
cat_2 = np.random.choice(['true', 'false'], size=n_samples)

# Timestamp column
timestamps = pd.date_range(start='2023-01-01', periods=1000, freq='D')
time = np.random.choice(timestamps, size=n_samples, replace=False)

# Multicategorical column
categories = ['A', 'B', 'C', 'D']
multi_1 = [random.sample(categories, k=random.randint(0, len(categories))) for _ in range(n_samples)]
categories = ['I', 'II', 'III']
multi_2 = [random.sample(categories, k=random.randint(0, len(categories))) for _ in range(n_samples)]

# Embedding column (assuming an embedding size of 5 for simplicity)
emb_1 = np.random.rand(n_samples, 3)
emb_2 = np.random.rand(n_samples, 2)

# Create the DataFrame
df = pd.DataFrame({
    'num_1': num_1,
    'num_2': num_2,
    'num_3': num_2,  # Duplicate column for testing
    'cat_1': cat_1,
    'cat_2': cat_2,
    'cat_3': cat_2,  # Duplicate column for testing
    'date': time,
    # 'multi_1': multi_1,
    # 'multi_2': multi_2,
    # 'multi_3': multi_2,  # Duplicate column for testing
    'emb_1': list(emb_1),
    'emb_2': list(emb_2),
    'emb_3': list(emb_2)  # Duplicate column for testing
})


dataset = Dataset(
    df,
    col_to_stype={
        'num_1': stype.numerical,
        'num_2': stype.numerical,
        'num_3': stype.numerical,  # Duplicate column
        'cat_1': stype.categorical,
        'cat_2': stype.categorical,
        'cat_3': stype.categorical,  # Duplicate column
        'date': stype.timestamp,
        # 'multi_1': stype.multicategorical,
        # 'multi_2': stype.multicategorical,
        # 'multi_3': stype.multicategorical,  # Duplicate column
        'emb_1': stype.embedding,
        'emb_2': stype.embedding,
        'emb_3': stype.embedding  # Duplicate column
    }
)

dataset.materialize()
tf = dataset.tensor_frame

tf.feat_dict[stype.timestamp].shape

torch.Size([10, 1, 7])

In [3]:
data = HeteroData()
data['paper'].tf = tf
data['author'].tf = tf # another tensor frame for authors
# Create an edge type "(author, writes, paper)":
data['author', 'writes', 'paper'].edge_index = torch.tensor([[0, 0, 1, 2, 2, 2], [0, 1, 2, 3, 4, 5]], dtype=torch.long)
data['paper', 'inv_writes', 'author'].edge_index = data['author', 'writes', 'paper'].edge_index.flip(0)  # Inverse edge type

### Database instance perturbations:

*Compute D'* : 

1. Modify the dataframes in the `Database()` object
2. Modify the graph instance directly, 
    - perturb features in the tensorframe
    - perturb pkfk pairs by rewiring / removing / adding edges

In [6]:
from src.explain.explain_utils import node_type_to_col_names, perturb_instance

In [7]:
# Test column masking
node_to_col_names = node_type_to_col_names(data)
masked_elements = [(node_type, col_name) for node_type in data.node_types for col_name in node_to_col_names[node_type]]
column_mask = {(node_type, col_name): torch.tensor([False]) for node_type, col_name in masked_elements}
column_mask[('paper', 'num_1')] = torch.tensor([True])
column_mask[('paper', 'cat_1')] = torch.tensor([True])
column_mask[('paper', 'emb_1')] = torch.tensor([True])

# Test row masking
row_mask = {node_type: torch.tensor([True if i%2==0 else False for i in range(data[node_type].tf.num_rows)]) for node_type in data.node_types}

# Test fkpk masking
fkpk_mask = {edge_type: torch.tensor([False]) for edge_type in data.edge_types}

In [8]:
# instance_perturbed = perturb_instance(data, column_mask, mask_type='column', perturbation_type='permutation_joint') # 'permutation_independent' 
instance_perturbed = perturb_instance(data, row_mask, mask_type='row', perturbation_type='permutation_independent') # 'permutation_independent' 

focus_type = embedding_type # numerical_type, categorical_type, time_type, embedding_type
print(instance_perturbed['paper'].tf.feat_dict[focus_type])
print(instance_perturbed['paper'].tf.feat_dict[focus_type][:, 0].values)
print(instance_perturbed['paper'].tf.feat_dict[focus_type][:, 1].values)
print()
print()
print(data['paper'].tf.feat_dict[focus_type])
print(data['paper'].tf.feat_dict[focus_type][:, 0].values)
print(data['paper'].tf.feat_dict[focus_type][:, 1].values)

MultiEmbeddingTensor(num_rows=10, num_cols=3, device='cpu')
tensor([[0.3731, 0.3818, 0.1581],
        [0.8010, 0.2583, 0.5532],
        [0.5733, 0.9530, 0.7581],
        [0.4514, 0.9376, 0.7369],
        [0.7703, 0.9887, 0.2196],
        [0.5616, 0.6680, 0.2141],
        [0.2281, 0.7262, 0.9031],
        [0.9713, 0.3643, 0.1491],
        [0.8398, 0.4882, 0.9242],
        [0.3556, 0.6223, 0.6211]])
tensor([[0.8393, 0.5897],
        [0.7942, 0.3066],
        [0.7955, 0.3217],
        [0.1224, 0.8300],
        [0.1585, 0.1847],
        [0.4398, 0.5238],
        [0.6832, 0.7638],
        [0.2236, 0.2363],
        [0.4667, 0.8972],
        [0.7807, 0.6489]])


MultiEmbeddingTensor(num_rows=10, num_cols=3, device='cpu')
tensor([[0.3731, 0.3818, 0.1581],
        [0.8010, 0.2583, 0.5532],
        [0.5733, 0.9530, 0.7581],
        [0.4514, 0.9376, 0.7369],
        [0.7703, 0.9887, 0.2196],
        [0.5616, 0.6680, 0.2141],
        [0.2281, 0.7262, 0.9031],
        [0.9713, 0.3643, 0.1491],
    

In [15]:
print(data.edge_index_dict[('author','writes','paper')])
# foreign_key_permutation foreign_key_exchange
instance_perturbed = perturb_instance(data, fkpk_mask, mask_type='fkpk', perturbation_type='foreign_key_exchange')
print(data.edge_index_dict[('author','writes','paper')])

tensor([[0, 0, 2, 1, 1, 1],
        [0, 1, 2, 3, 4, 5]])
tensor([[1, 1, 0, 2, 2, 2],
        [0, 1, 2, 3, 4, 5]])
